# Qwen 2.5 Intent Distillation Pipeline
Run this notebook on Google Colab (T4 GPU recommended).

This notebook covers:
1. Setup & Installations
2. Dataset Preparation
3. Teacher Labeling (Qwen2.5-7B-Instruct)
4. Student Fine-Tuning (Qwen2.5-0.5B-Instruct via QLoRA)
5. Model Evaluation
6. CPU Benchmark

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Đổi thư mục làm việc vào thư mục bạn muốn chứa dự án trên Drive
%cd /content/drive/MyDrive/

# 2. Clone repo từ GitHub về (Vì đây là repo Public nên bạn chỉ cần chạy URL bình thường)
!git clone https://github.com/kienng221105/Voice_Interaction_For_Robot.git

# 3. Di chuyển vào thư mục dự án
%cd Voice_Interaction_For_Robot


Mounted at /content/drive
/content/drive/MyDrive
fatal: destination path 'Voice_Interaction_For_Robot' already exists and is not an empty directory.
/content/drive/MyDrive/Voice_Interaction_For_Robot


In [2]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes scikit-learn psutil tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 18.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.4 MB/s eta 0:00:0000:01:00:01


## 1. Dataset Preparation
Converts raw datasets to `{"text": "...", "label": "intent_name"}` format and splits into train/test sets.

In [3]:
import json
import random
import os
import torch

os.makedirs('data/qwen_distill', exist_ok=True)
os.makedirs('reports', exist_ok=True)

gold_samples = []
unlabeled_samples = []

with open("data/raw/dataset_final.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        if d.get("type") == "unlabeled":
            unlabeled_samples.append({"text": d["text"]})
        else:
            gold_samples.append({"text": d["text"], "label": d["intent"]})

hard_test_samples = []
with open("data/raw/dataset_150_bonus_v2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        hard_test_samples.append({"text": d["text"], "label": d["intent"]})

random.seed(42)
random.shuffle(gold_samples)
normal_test_samples = gold_samples[:50]
train_gold_samples = gold_samples[50:]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(train_gold_samples, "data/qwen_distill/train_gold.jsonl")
save_jsonl(unlabeled_samples, "data/qwen_distill/unlabeled.jsonl")
save_jsonl(normal_test_samples, "data/qwen_distill/normal_test.jsonl")
save_jsonl(hard_test_samples, "data/qwen_distill/hard_test.jsonl")

print(f"Train Gold: {len(train_gold_samples)}")
print(f"Unlabeled: {len(unlabeled_samples)}")
print(f"Normal Test: {len(normal_test_samples)}")
print(f"Hard Test: {len(hard_test_samples)}")

Train Gold: 150
Unlabeled: 800
Normal Test: 50
Hard Test: 150


## 2. Teacher Prompt Setup

In [4]:
system_prompt = """You are an expert intent classifier for a Vietnamese vending machine assistant.
Analyze the user's input text (which may contain ASR noise, e.g., "xtinh" -> "sting") and output ONLY the corresponding intent name from the Valid Intents list. Do not output JSON or any other text.

Valid Intents:
- greeting
- show_menu
- buy_product
- add_product
- change_product
- payment
- cancel
- help
- unknown

Example 1:
Text: "cho tôi một lon coca"
Intent: buy_product

Example 2:
Text: "à thôi hủy đi"
Intent: cancel
"""

with open("teacher_prompt.txt", "w", encoding="utf-8") as f:
    f.write(system_prompt)

## 3. Teacher Pseudo-Labeling
We use **Qwen2.5-7B-Instruct** loaded in 4-bit to label the 800 unlabeled samples.

In [5]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.notebook import tqdm

torch.cuda.empty_cache()
gc.collect()

teacher_model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading Teacher Model {teacher_model_name}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)
teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16 # Khắc phục lỗi BFloat16 trên T4
)

intents = ["greeting", "show_menu", "buy_product", "add_product", "change_product", "payment", "cancel", "help", "unknown"]
pseudo_labeled = []

for item in tqdm(unlabeled_samples, desc="Teacher Labeling"):
    text = item["text"]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Text: \"{text}\"\nIntent:"}
    ]
    
    text_input = teacher_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = teacher_tokenizer([text_input], return_tensors="pt").to(teacher_model.device)
    
    generated_ids = teacher_model.generate(**model_inputs, max_new_tokens=10, do_sample=False)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = teacher_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    predicted = "unknown"
    for intent in intents:
        if intent in response:
            predicted = intent
            break
            
    pseudo_labeled.append({"text": text, "label": predicted})

save_jsonl(pseudo_labeled, "data/qwen_distill/pseudo_labeled.jsonl")

# Merge gold + pseudo
train_distill = train_gold_samples + pseudo_labeled
save_jsonl(train_distill, "data/qwen_distill/train_distill.jsonl")
print(f"Created train_distill.jsonl with {len(train_distill)} samples.")

# Free up VRAM for student training
del teacher_model
del teacher_tokenizer
torch.cuda.empty_cache()
gc.collect()

Loading Teacher Model Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher Labeling:   0%|          | 0/800 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Created train_distill.jsonl with 950 samples.


4229

## 4. Student Fine-Tuning (Qwen2.5-0.5B-Instruct via QLoRA)
# Loại bỏ hoàn toàn BFloat16 do lỗi của transformers/peft
student_model.config.torch_dtype = torch.float16
for param in student_model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)



In [6]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

student_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Format Dataset for Causal LM training
formatted_data = []
for item in train_distill:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Text: \"{item['text']}\"\nIntent:"},
        {"role": "assistant", "content": item['label']}
    ]
    formatted_data.append({"messages": messages})
    
train_dataset = Dataset.from_list(formatted_data)

student_tokenizer = AutoTokenizer.from_pretrained(student_model_name)
student_tokenizer.pad_token = student_tokenizer.eos_token

def apply_chat_template(example):
    example['text'] = student_tokenizer.apply_chat_template(example['messages'], tokenize=False)
    return example
    
train_dataset = train_dataset.map(apply_chat_template)

# Khởi tạo Base Model với fp16 thay vì bfloat16
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# KHÔNG GỌI get_peft_model() BẰNG TAY Ở ĐÂY!
# Hãy để SFTTrainer tự động bọc PeftModel thông qua peft_config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# Khắc phục lỗi SFTConfig trên thư viện trl mới nhất
training_args = SFTConfig(
    dataset_text_field="text",
    output_dir="./student_qwen",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_32bit",
    fp16=True,
    report_to="none"
    # Bỏ max_seq_length hoàn toàn để tương thích chéo các bản TRL
)

trainer = SFTTrainer(
    model=student_model,
    train_dataset=train_dataset,
    processing_class=student_tokenizer,
    args=training_args,
    peft_config=peft_config
)

print("Starting student fine-tuning...")
trainer.train()

trainer.model.save_pretrained("./student_qwen_adapter")
student_tokenizer.save_pretrained("./student_qwen_adapter")
print("Training complete. Adapter saved.")

# Clear memory
del trainer
del student_model
torch.cuda.empty_cache()
gc.collect()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

/tmp/ipykernel_1023/2082920976.py:49: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Tokenizing train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting student fine-tuning...


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

## 5. Model Evaluation (Accuracy, F1)
We will evaluate the Student Model on Normal and Hard test sets.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from peft import PeftModel

def evaluate_model(model, tokenizer, data, system_prompt, model_name, dataset_name):
    intents = ["greeting", "show_menu", "buy_product", "add_product", "change_product", "payment", "cancel", "help", "unknown"]
    y_true = []
    y_pred = []
    
    for item in tqdm(data, desc=f"Evaluating {model_name} on {dataset_name}"):
        text = item["text"]
        y_true.append(item["label"])
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Text: \"{text}\"\nIntent:"}
        ]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
        
        generated_ids = model.generate(**model_inputs, max_new_tokens=10, do_sample=False)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        predicted = "unknown"
        for intent in intents:
            if intent in response:
                predicted = intent
                break
        y_pred.append(predicted)
        
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    
    print(f"\n--- Results for {model_name} on {dataset_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    
    return acc, macro_f1

print("Loading Student Model for Evaluation...")
adapter_path = "./student_qwen_adapter"
eval_tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    student_model_name, 
    quantization_config=bnb_config, 
    device_map="auto",
    torch_dtype=torch.float16  # Khắc phục lỗi tương tự khi load model test
)
eval_model = PeftModel.from_pretrained(base_model, adapter_path)

evaluate_model(eval_model, eval_tokenizer, normal_test_samples, system_prompt, "Student (0.5B)", "Normal Test")
evaluate_model(eval_model, eval_tokenizer, hard_test_samples, system_prompt, "Student (0.5B)", "Hard Test")

del eval_model
del base_model
torch.cuda.empty_cache()
gc.collect()

## 6. CPU Edge Benchmark
Measures inference latency and RAM usage for deploying on Vending Machine hardware (CPU only).

In [ ]:
import time
import psutil

def measure_inference(model, tokenizer, text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Text: \"{text}\"\nIntent:"}
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    
    start_time = time.time()
    _ = model.generate(**model_inputs, max_new_tokens=10, do_sample=False)
    end_time = time.time()
    return end_time - start_time

print("Loading Student Model on CPU...")
start_load = time.time()
# Đối với CPU, chạy torch.float32 (Native Type) sẽ an toàn hơn
cpu_base = AutoModelForCausalLM.from_pretrained(student_model_name, device_map="cpu", torch_dtype=torch.float32)
cpu_model = PeftModel.from_pretrained(cpu_base, "./student_qwen_adapter")
cpu_tokenizer = AutoTokenizer.from_pretrained("./student_qwen_adapter")
end_load = time.time()

process = psutil.Process()
memory_info = process.memory_info()
mem_mb = memory_info.rss / (1024 * 1024)

print(f"\n--- Student CPU Benchmark ---")
print(f"Load Time: {end_load - start_load:.2f} seconds")
print(f"Memory Usage (RSS): {mem_mb:.2f} MB")

# Warmup
measure_inference(cpu_model, cpu_tokenizer, "cho tôi một lon coca")

test_texts = ["mua 1 chai nước suối", "thôi không mua nữa", "có nước gì ngon không", "thanh toán momo"]
latencies = []
for t in test_texts:
    latencies.append(measure_inference(cpu_model, cpu_tokenizer, t))
    
print(f"Average CPU Inference Latency: {sum(latencies)/len(latencies):.4f} seconds/query")